# 🚀 t4-diffusion: Stable Diffusion on Free Colab T4

This notebook demonstrates running Stable Diffusion on Google Colab's free T4 GPU with optimizations.

**Current Features:**
- FP16 inference with memory optimizations
- Feature caching for inference acceleration
- VRAM monitoring with 15.6GB T4 limit enforcement
- Engine save/load for faster subsequent runs

**Note on TensorRT/INT8:**
As of March 2026, Google Colab uses CUDA 13.x which has compatibility issues with TensorRT packages (built for CUDA 12.x). INT8 quantization and TensorRT compilation are automatically skipped when these dependencies aren't available. The pipeline falls back to optimized FP16 inference.

**Requirements:**
- Google Colab with T4 GPU runtime
- ~10GB free VRAM

## 1. Setup & Installation

In [ ]:
# Install t4-diffusion from GitHub
!pip install git+https://github.com/Kash6/t4-diffusion.git -q --no-cache-dir
!pip install hypothesis pytest -q  # For running tests

In [ ]:
# Verify GPU and CUDA environment
import torch
import subprocess

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cuda_version = torch.version.cuda
    print(f"✓ GPU: {gpu_name}")
    print(f"✓ VRAM: {total_vram:.1f} GB")
    print(f"✓ CUDA: {cuda_version}")
    print(f"✓ PyTorch: {torch.__version__}")
    
    if "T4" in gpu_name:
        print("✓ Running on T4 GPU (optimal for this package)")
else:
    print("❌ No GPU detected!")
    print("   Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Check TensorRT availability
trt_available = False
modelopt_available = False

try:
    import tensorrt
    trt_available = True
    print(f"✓ TensorRT: {tensorrt.__version__}")
except ImportError:
    print("⚠ TensorRT not available (CUDA version mismatch with Colab)")

try:
    import modelopt
    modelopt_available = True
    print(f"✓ nvidia-modelopt available")
except ImportError:
    print("⚠ nvidia-modelopt not available")

if not trt_available or not modelopt_available:
    print("\n📝 Pipeline will use optimized FP16 inference (no INT8/TensorRT)")
    print("   This is expected on current Colab due to CUDA 13.x compatibility.")

## 2. Basic Inference with SDXL-Turbo

In [ ]:
from diffusion_trt.pipeline import PipelineConfig, OptimizedPipeline
from diffusion_trt.utils.vram_monitor import VRAMMonitor

# Configure the pipeline
config = PipelineConfig(
    model_id="stabilityai/sdxl-turbo",
    enable_int8=True,           # Will be skipped if deps unavailable
    enable_caching=True,        # Enable feature caching
    num_inference_steps=4,      # SDXL-Turbo uses 4 steps
    guidance_scale=0.0,         # No CFG for SDXL-Turbo
    seed=42,                    # For reproducible results
)

print("Configuration:")
print(f"  Model: {config.model_id}")
print(f"  INT8: {config.enable_int8} (if available)")
print(f"  Caching: {config.enable_caching}")
print(f"  Steps: {config.num_inference_steps}")

In [ ]:
# Load and optimize the model
print("Loading model...")
print("This may take a few minutes on first run (downloading weights).")

with VRAMMonitor() as monitor:
    pipeline = OptimizedPipeline.from_pretrained(
        config.model_id,
        config=config,
    )

print(f"\n✓ Model loaded!")
print(f"✓ Peak VRAM: {monitor.peak_gb:.2f} GB")

In [ ]:
# Generate an image
prompt = "A beautiful sunset over a calm ocean, vibrant colors, photorealistic"

print(f"Generating: '{prompt}'")

with VRAMMonitor() as monitor:
    images = pipeline(prompt)

print(f"\n✓ Generated!")
print(f"✓ VRAM used: {monitor.peak_gb:.2f} GB")

# Display the image
images[0]

## 3. VRAM Monitoring

In [ ]:
from diffusion_trt.utils.vram_monitor import get_vram_usage, get_peak_vram, clear_cache
from diffusion_trt.models import T4_VRAM_LIMIT_GB

print(f"T4 VRAM Limit: {T4_VRAM_LIMIT_GB} GB")
print(f"Current VRAM: {get_vram_usage():.2f} GB")
print(f"Peak VRAM: {get_peak_vram():.2f} GB")
print(f"Available: {T4_VRAM_LIMIT_GB - get_vram_usage():.2f} GB")

In [ ]:
# Generate multiple images with VRAM monitoring
prompts = [
    "A majestic mountain landscape with snow-capped peaks",
    "A futuristic cityscape at night with neon lights",
    "A cozy cabin in the woods during autumn",
]

for i, prompt in enumerate(prompts, 1):
    with VRAMMonitor() as monitor:
        images = pipeline(prompt)
    
    print(f"Image {i}: VRAM={monitor.peak_gb:.2f}GB | '{prompt[:40]}...'")
    display(images[0])

## 4. Benchmark

In [ ]:
# Run benchmark
print("Running benchmark (5 iterations + 2 warmup)...")

metrics = pipeline.benchmark(
    prompt="A beautiful landscape with mountains and a lake",
    num_iterations=5,
    warmup_iterations=2,
)

print("\n" + "="*50)
print("BENCHMARK RESULTS (FP16 Baseline)")
print("="*50)
print(f"Latency (mean):  {metrics.latency_mean_ms:.0f} ms")
print(f"Latency (std):   {metrics.latency_std_ms:.0f} ms")
print(f"Latency (p50):   {metrics.latency_p50_ms:.0f} ms")
print(f"Latency (p95):   {metrics.latency_p95_ms:.0f} ms")
print(f"Throughput:      {metrics.throughput_images_per_sec:.2f} img/s")
print(f"Peak VRAM:       {metrics.vram_peak_gb:.2f} GB")
print(f"Cache Hit Rate:  {metrics.cache_hit_rate:.1%}")

## 5. Batch Inference

In [ ]:
# Generate multiple images with different prompts
batch_prompts = [
    "A cyberpunk street scene with rain and neon signs",
    "An ancient temple hidden in a misty forest",
    "A steampunk airship flying over Victorian London",
    "A magical library with floating books and glowing orbs",
]

print(f"Generating {len(batch_prompts)} images...")

import time
start = time.time()

batch_images = []
for prompt in batch_prompts:
    images = pipeline(prompt)
    batch_images.extend(images)

total_time = time.time() - start
avg_time = total_time / len(batch_prompts)

print(f"\n✓ Generated {len(batch_images)} images")
print(f"✓ Total time: {total_time:.1f}s")
print(f"✓ Average: {avg_time:.1f}s per image")
print(f"✓ Throughput: {len(batch_images)/total_time:.2f} img/s")

In [ ]:
# Display batch results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, img, prompt in zip(axes.flat, batch_images, batch_prompts):
    ax.imshow(img)
    ax.set_title(prompt[:40] + "...", fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Cleanup

In [ ]:
# Clear all caches and free memory
pipeline.clear_cache()
clear_cache()

import gc
gc.collect()
torch.cuda.empty_cache()

print(f"Final VRAM usage: {get_vram_usage():.2f} GB")

---

## Summary

This notebook demonstrated:

1. **Installation** - Simple pip install from GitHub
2. **Basic Inference** - Generate images with SDXL-Turbo (~1.5s per image)
3. **VRAM Monitoring** - Track memory usage to stay within T4 limits
4. **Benchmarking** - Measure latency and throughput
5. **Batch Inference** - Generate multiple images efficiently

**Current Performance (FP16 Baseline on T4):**
- ~1.5-1.6s per image at 512×512
- ~0.6-0.7 images/second throughput
- ~12 GB peak VRAM

**Note:** INT8/TensorRT optimizations are not currently available on Colab due to CUDA version compatibility. When these become available, expect ~1.5-2x speedup.

For more information, visit: https://github.com/Kash6/t4-diffusion